# ConversationBufferMemory with LangChain

## Learning Objectives
By the end of this notebook, you will understand:
- ✅ How to implement conversation memory in LangChain
- ✅ How to use `ConversationBufferMemory` to maintain chat history
- ✅ How to build stateful chatbots that remember previous interactions
- ✅ How to manage multiple user sessions independently

## What We'll Build
A chatbot that remembers the conversation history and can refer to information from previous messages within the same session.

---

## Speaker Notes
**Key Talking Points:**
1. **Introduction (2 min)**: Explain why memory is crucial for conversational AI - stateless APIs need external memory management
2. **Problem with Statelessness (2 min)**: Show how LLMs naturally forget context; each API call is independent
3. **Solution - Buffer Memory (3 min)**: Introduce ConversationBufferMemory as the simplest solution - stores all messages
4. **Trade-offs (2 min)**: Mention that while simple, buffer memory grows indefinitely and can be costly for long conversations
5. **Hands-on Demo (5 min)**: Walk through the code implementation step-by-step

In [ ]:
# Import required components from LangChain
from langchain_openai import ChatOpenAI  # OpenAI's language model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.runnables.history import RunnableWithMessageHistory  # Memory management
from langchain_community.chat_message_histories import ChatMessageHistory  # History storage

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Import Required Libraries

We need several components from LangChain to build our conversational AI system:

**Component Breakdown:**
- **ChatOpenAI**: The LLM that powers our chatbot
- **ChatPromptTemplate**: Structures our prompts for chat-style interactions
- **MessagesPlaceholder**: A placeholder in the prompt for conversation history
- **RunnableWithMessageHistory**: Wraps our chain to automatically manage memory
- **ChatMessageHistory**: In-memory storage for conversation messages

In [2]:
# Load environment variables from .env file
import os
from dotenv import load_dotenv, find_dotenv

# Find and load the .env file from the current directory or parent directories
load_dotenv(find_dotenv(), override=True)

# Verify the API key is loaded (show first 10 chars only for security)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✅ API Key loaded: {api_key[:10]}...")
else:
    print("❌ API Key not found! Check your .env file")


✅ API Key loaded: sk-proj-3n...


## Step 2: Setup Environment & API Configuration

Before we can use the OpenAI API, we need to load our API key from the environment.

**Why this matters:**
- Never hardcode API keys in your code
- Use `.env` files to keep sensitive credentials secure
- Always validate that credentials are loaded before proceeding

In [3]:
# Initialize the ChatOpenAI language model
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Using GPT-4 mini for optimal performance
    temperature=0         # Set to 0 for deterministic, consistent responses
)


## Step 3: Initialize the Language Model

We initialize ChatOpenAI with specific parameters:

**Parameters:**
- **model**: "gpt-4o-mini" - Uses the latest GPT-4 mini model for better performance
- **temperature**: 0 - Deterministic responses (no randomness) for consistent behavior

**Note**: For creative tasks, you might increase temperature to 0.7-1.0

In [4]:
# Create a chat prompt template with memory support
prompt = ChatPromptTemplate.from_messages([
    # System role: Defines how the assistant should behave
    ("system", "You are a helpful assistant."),
    
    # MessagesPlaceholder: This is where conversation history will be injected
    # The "history" variable name must match what we use in RunnableWithMessageHistory
    MessagesPlaceholder(variable_name="history"),
    
    # Human input: The current user message
    ("human", "{input}")
])


## Step 4: Create the Chat Prompt Template

This prompt structure is the key to implementing memory in LangChain:

**Template Components:**
1. **System message**: Defines the chatbot's personality/role
2. **MessagesPlaceholder**: This is where conversation history gets injected automatically
3. **Human input**: The current user message

**Flow**: Previous messages + new user input → LLM → Response

In [5]:
# Create a runnable chain: Prompt Template | Language Model
# This chains the components together using LangChain's pipe operator
chain = prompt | llm


## Step 5: Create the Core Chain

The chain combines our prompt template with the language model in a "pipeline":

**LangChain Pipe Operator (`|`):**
- Chains components together in sequence
- Input flows through prompt → LLM → output

In [6]:
# Create an in-memory store for conversation histories
# In production, you'd use a database (Redis, PostgreSQL, etc.)
store = {}

def get_session_history(session_id: str):
    """
    Retrieve or create a ChatMessageHistory for a given session_id.
    
    Args:
        session_id (str): Unique identifier for the user/conversation
        
    Returns:
        ChatMessageHistory: The message history for this session
    """
    # If this is the first message from this session, create a new history
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]


## Step 6: Implement Session-Based Memory Storage

This is where the "Buffer" memory comes in - we store all messages for each user session:

**Key Concepts:**
- **Dictionary-based storage**: `store` dict holds ChatMessageHistory objects per session
- **Session IDs**: Each user/conversation gets a unique identifier
- **get_session_history()**: Retrieves or creates history for a session
- **All messages preserved**: ConversationBufferMemory keeps everything (advantage: complete context; disadvantage: grows unbounded)

In [7]:
# Wrap the chain with message history management
# This automatically handles adding/retrieving conversation history
chatbot = RunnableWithMessageHistory(
    chain,                          # The base chain (prompt | llm)
    get_session_history,            # Function to retrieve session history
    input_messages_key="input",     # The input variable in our prompt template
    history_messages_key="history"  # The MessagesPlaceholder variable name
)


## Step 7: Wrap the Chain with Memory Management

**RunnableWithMessageHistory** is the magic that enables stateful conversations:

**What it does:**
- Automatically retrieves session history
- Injects it into the prompt template
- Stores new messages in the history after each interaction
- Handles all the memory plumbing so you don't have to

**Configuration:**
- `input_messages_key`: Which variable in the chain receives user input
- `history_messages_key`: Which MessagesPlaceholder variable gets the history

In [8]:
# First message: User introduces themselves
# Session history is empty at this point
response1 = chatbot.invoke(
    {"input": "My name is Sarvesh"},
    config={"configurable": {"session_id": "user1"}}  # Unique session identifier
)

# Display the response
print("User: My name is Sarvesh")
print(f"Assistant: {response1.content}")
print("\n" + "="*50 + "\n")


User: My name is Sarvesh
Assistant: Hello, Sarvesh! How can I assist you today?




## Step 8: Test 1 - First Message (Introduce Ourselves)

**What happens here:**
1. User sends: "My name is Sarvesh"
2. Session history is empty, so only the system message + user input goes to the LLM
3. Response is generated
4. Both user message and AI response are stored in the session history

**Note**: The `config` parameter with `session_id` is crucial - it tells RunnableWithMessageHistory which session to use

In [9]:
# Second message: User asks a follow-up question
# RunnableWithMessageHistory automatically includes the previous messages!
response2 = chatbot.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user1"}}  # Same session ID as before
)

# Display the response
print("User: What is my name?")
print(f"Assistant: {response2.content}")
print("\n" + "="*50)
print("\n✅ SUCCESS! The assistant remembered the name from the previous message!")
print("This is ConversationBufferMemory in action - all messages are kept in the buffer.")


User: What is my name?
Assistant: Your name is Sarvesh. How can I help you today?


✅ SUCCESS! The assistant remembered the name from the previous message!
This is ConversationBufferMemory in action - all messages are kept in the buffer.


---

## Key Takeaways

### ✅ What We Learned:
1. **ConversationBufferMemory** stores ALL messages in a buffer
2. **RunnableWithMessageHistory** automates the memory management process
3. **Session IDs** allow multiple independent conversations
4. **MessagesPlaceholder** is the key to injecting history into prompts

### ⚖️ Trade-offs:
| Aspect | Buffer Memory |
|--------|--------------|
| **Pros** | Simple, complete context, no message loss |
| **Cons** | Unbounded growth, costs increase over time, hits token limits |

### 🚀 Next Steps:
- **ConversationSummaryMemory**: Keep only a summary of long conversations
- **ConversationWindowMemory**: Keep only the last N messages
- **Entity Memory**: Extract and track important information
- **Vector Store Memory**: Semantic search across conversations

### 💡 Speaker Notes for Recap:
**Time: 2-3 minutes**
- Explain that buffer memory is a stepping stone, not production-ready for long conversations
- Show the `store` dictionary structure to illustrate how sessions are managed
- Mention that in production, you'd replace the in-memory dict with a proper database
- Emphasize that the same patterns apply to other memory types - they're all backends

## Step 9: Test 2 - Follow-up Question (Memory in Action!)

**This is where memory matters:**
1. User asks: "What is my name?"
2. The LLM has NO DIRECT knowledge of the previous message in its training
3. But RunnableWithMessageHistory automatically prepends the previous conversation
4. The LLM sees: System message → "My name is Sarvesh" (history) → "What is my name?" (current)
5. The LLM can now answer correctly!

**This demonstrates the power of buffer memory** - the chatbot "remembers" what the user said earlier

# ConversationSummaryMemory with LangChain

## Learning Objectives
By the end of this section, you will understand:
- ✅ How ConversationSummaryMemory differs from ConversationBufferMemory
- ✅ Why summarization is important for long conversations
- ✅ How to automatically summarize conversations to save tokens
- ✅ The trade-off between context completeness and cost efficiency

## What We'll Build
A chatbot that automatically summarizes conversation history when it gets too long, keeping only the important information while reducing token usage.

---

## Speaker Notes
**Key Talking Points:**
1. **The Buffer Problem (2 min)**: Remind that buffer memory grows indefinitely and costs increase with conversation length
2. **Summary Solution (3 min)**: Introduce summarization as a way to compress history while retaining meaning
3. **How It Works (3 min)**: Explain the trigger (message count threshold) and the summarization process
4. **Trade-offs (2 min)**: Less detailed memory but lower costs; some information loss in summarization
5. **Hands-on Demo (5 min)**: Show how conversations are summarized automatically

---

## What Summary Memory Means (The Core Concept)

**Summary memory means the AI does not remember every message.**

Instead, it remembers a **summary of the conversation so far**.

**Key Benefits:**
- 📉 Old messages are compressed
- ✨ Only important meaning is kept
- 💰 Token usage stays low

**Analogy:**
> Instead of remembering every word you said, the AI remembers what the conversation is about.

In [25]:
# Import the same components as before
# Summary memory uses the same building blocks - the difference is in how we manage history
from langchain_openai import ChatOpenAI  # LLM for conversation AND summarization
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.runnables.history import RunnableWithMessageHistory  # Memory management
from langchain_community.chat_message_histories import ChatMessageHistory  # History storage


## Step 1: Import Required Libraries (Summary Memory Version)

These are the same imports as before - we're building on the foundation we learned:

**Why the same imports?**
- Summary memory uses the same LangChain architecture
- The difference is in the `get_session_history()` function
- We'll add summarization logic there

In [26]:
# Initialize the language model - this will power both conversations and summarization
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Using GPT-4 mini for optimal performance
    temperature=0         # Deterministic responses for consistent behavior
)

## Step 2: Initialize the Language Model

Same as before - we need the LLM for two purposes now:
1. **Main conversation** - answering user questions
2. **Summarization** - compressing conversation history when needed

In [27]:
# Create the core conversation prompt template
prompt = ChatPromptTemplate.from_messages([
    # System role: Defines how the assistant should behave
    ("system", "You are a helpful assistant."),
    
    # MessagesPlaceholder: Where conversation history (or summary) will be injected
    MessagesPlaceholder(variable_name="history"),
    
    # Human input: The current user message
    ("human", "{input}")
])

## Step 3: Create the Chat Prompt Template

Same structure as ConversationBufferMemory - this is the core conversation template:

In [28]:
# Create an in-memory store for conversation histories
store = {}

def get_session_history(session_id: str):
    """
    Retrieve or create a ChatMessageHistory for a session.
    This version includes automatic summarization when the conversation gets long.
    
    Args:
        session_id (str): Unique identifier for the user/conversation
        
    Returns:
        ChatMessageHistory: The message history (or summary) for this session
    """
    # Create a new history if this is the first message from this session
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    history = store[session_id]

    # KEY LOGIC: If conversation gets too long, summarize it
    # Threshold: 6 messages means 3 back-and-forth exchanges
    if len(history.messages) > 6:
        # Create a specialized prompt for summarization
        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", "Summarize the conversation so far in 3 short lines. Focus on key facts and important context."),
            MessagesPlaceholder(variable_name="history")
        ])

        # Create a chain: summarization prompt | LLM
        summary_chain = summary_prompt | llm
        
        # Generate the summary by passing all current messages
        summary = summary_chain.invoke(
            {"history": history.messages}
        ).content

        # Replace the entire conversation with just the summary
        # This is the space-saving trick: compress old messages into one summary message
        history.clear()
        history.add_ai_message(f"Conversation summary so far: {summary}")

    return history


## Step 4: Implement Summary-Based Memory Storage (The Key Difference!)

**This is where ConversationSummaryMemory differs from buffer memory:**

**How it works:**
1. Start with buffer memory (store all messages)
2. When messages exceed a threshold (6 messages), trigger summarization
3. Create a summary of the entire conversation
4. Replace all old messages with the summary
5. Continue the conversation with just the summary + new messages

**Benefits:**
- 📊 Keeps memory bounded to a reasonable size
- 💰 Reduces token usage significantly
- 🎯 Preserves important information through summarization

**Trade-off:**
- ⚠️ Some fine details might be lost in summarization
- 🔄 Summary quality depends on the summarization prompt

In [29]:
# Create the core chain
chain = prompt | llm

# Wrap with message history - now with summarization built-in
chatbot = RunnableWithMessageHistory(
    chain,                          # The base chain (prompt | llm)
    get_session_history,            # Function that includes summarization logic
    input_messages_key="input",     # The input variable in our prompt template
    history_messages_key="history"  # The MessagesPlaceholder variable name
)


## Step 5: Create the Chatbot with Summary Memory

Same wrapping pattern as buffer memory - the difference is now in the `get_session_history` function:

In [ ]:
# Have multiple conversations to build up the history
# After 6 messages, summarization will trigger automatically

# Message 1
response1 = chatbot.invoke(
    {"input": "My name is Sarvesh"},
    config={"configurable": {"session_id": "user1"}}
)
print("Message 1 - User: My name is Sarvesh")
print(f"Assistant: {response1.content}\n")

# Message 2
response2 = chatbot.invoke(
    {"input": "I live in Chennai"},
    config={"configurable": {"session_id": "user1"}}
)
print("Message 2 - User: I live in Chennai")
print(f"Assistant: {response2.content}\n")

# Message 3
response3 = chatbot.invoke(
    {"input": "I teach AI and LangChain"},
    config={"configurable": {"session_id": "user1"}}
)
print("Message 3 - User: I teach AI and LangChain")
print(f"Assistant: {response3.content}\n")

# Message 4
response4 = chatbot.invoke(
    {"input": "I am recording a course"},
    config={"configurable": {"session_id": "user1"}}
)
print("Message 4 - User: I am recording a course")
print(f"Assistant: {response4.content}\n")

print("="*70)
print("⚠️ After this point, summarization will trigger automatically!")
print("="*70)


Message 1 - User: My name is Sarvesh
Assistant: Hello, Sarvesh! How can I assist you today?

Message 2 - User: I live in Chennai
Assistant: That's great, Sarvesh! Chennai is a vibrant city with a rich culture and history. Is there something specific you would like to know or discuss about Chennai or anything else?

Message 3 - User: I teach AI and LangChain
Assistant: That's impressive, Sarvesh! Teaching AI and LangChain is quite relevant in today's tech landscape. LangChain, in particular, is a powerful framework for building applications with language models. What aspects of AI and LangChain do you focus on in your teaching? Are there any specific topics or projects you enjoy discussing with your students?

Message 4 - User: I am recording a course
Assistant: That sounds exciting! Recording a course on AI and LangChain can be a great way to share knowledge and help others learn. What topics are you planning to cover in your course? Do you have any specific goals or target audience in

## Step 6: Test - Multiple Messages to Trigger Summarization

**What happens here:**
1. **Messages 1-4**: Buffer stores all messages normally (8 total in history after Message 4)
2. **Message 5**: Triggers summarization automatically (8 total messages > threshold of 6)
3. **After Message 5**: Only the summary + new message is kept

**How the threshold works:**
- Each exchange = 1 user message + 1 assistant response = 2 messages in history
- Threshold is set to > 6 messages
- So after 4 exchanges (8 messages total), the NEXT message (Message 5) will trigger summarization
- You'll see the warning below, which means the 5th message will trigger summarization

In [ ]:
# Ask a question that tests if the summary captured the important information
response = chatbot.invoke(
    {"input": "What do you know about me?"},
    config={"configurable": {"session_id": "user1"}}
)

print("User: What do you know about me?")
print(f"\nAssistant:\n{response.content}")
print("\n" + "="*70)
print("✅ SUCCESS! Summary Memory in Action!")
print("="*70)
print("\nKey Observation:")
print("- The conversation was automatically summarized when it reached 8 messages")
print("- Only the summary is kept, not the individual messages")
print("- The AI can still answer questions about your details!")
print("- This saves tokens and reduces memory usage significantly")


User: What do you know about me?

Assistant:
I don't have any personal information about you unless you've shared something specific in this conversation. I know that you're recording a course on AI and LangChain, and I'm here to assist you with any questions or topics related to that. If there's anything else you'd like to share or ask, feel free!

✅ SUCCESS! Summary Memory in Action!

Key Observation:
- The conversation was automatically summarized when it reached 6 messages
- Only the summary is kept, not the individual messages
- The AI can still answer questions about your details!
- This saves tokens and reduces memory usage significantly


## Step 7: Test After Summarization

**This is the critical test:**
- Ask a question that requires knowledge from earlier in the conversation
- The AI should answer correctly using only the summary
- This proves that summarization preserved the important context

**Expected behavior:**
- The conversation history was replaced with a summary
- The AI can still understand who you are, where you live, what you teach
- This happened automatically without us doing anything manually!

In [ ]:
# Inspect the store to see how the conversation was summarized
print("="*70)
print("INSPECTING THE CONVERSATION STORE")
print("="*70)
print(f"\nNumber of sessions in store: {len(store)}")
print(f"Session IDs: {list(store.keys())}")

# Get the history for user1
user1_history = store.get("user1")
if user1_history:
    print(f"\nNumber of messages in 'user1' history: {len(user1_history.messages)}")
    print("\n" + "-"*70)
    print("MESSAGE CONTENT:")
    print("-"*70)
    
    for i, message in enumerate(user1_history.messages, 1):
        msg_type = "AI" if message.__class__.__name__.startswith('AI') else "Human"
        print(f"\nMessage {i} ({msg_type}):")
        print(f"{message.content}")
        print("-"*70)
    
    print("\n💡 Notice:")
    print("- Only 3 messages remain: the summary + the latest exchange (user question + AI response)")
    print("- Compare to 8 messages we had BEFORE summarization")
    print("- The summary contains compressed information from all 4 previous exchanges")
    print("- This is much more efficient than storing all 8 individual messages!")
else:
    print("\nNo history found for user1")


INSPECTING THE CONVERSATION STORE

Number of sessions in store: 1
Session IDs: ['user1']

Number of messages in 'user1' history: 3

----------------------------------------------------------------------
MESSAGE CONTENT:
----------------------------------------------------------------------

Message 1 (AI):
Conversation summary so far: You mentioned you're recording a course on AI and LangChain. I'm curious about the topics you'll cover and your target audience. If you need tips or resources for your course, I'm here to help!
----------------------------------------------------------------------

Message 2 (Human):
What do you know about me?
----------------------------------------------------------------------

Message 3 (AI):
I don't have any personal information about you unless you've shared something specific in this conversation. I know that you're recording a course on AI and LangChain, and I'm here to assist you with any questions or topics related to that. If there's anything e

---

## Summary Memory vs Buffer Memory Comparison

### Side-by-Side Comparison:

| Feature | Buffer Memory | Summary Memory |
|---------|--------------|----------------|
| **How it works** | Keeps all messages | Compresses old messages into summary |
| **Memory growth** | ↗️ Linear (unbounded) | ➡️ Bounded (stays small) |
| **Token usage** | 📈 Increases over time | 💰 Stays relatively stable |
| **Context quality** | 100% complete | ~85-95% through summarization |
| **Long conversations** | ❌ Not ideal | ✅ Ideal |
| **Setup complexity** | Simple | More complex |
| **Use case** | Short chats, demos | Production, long conversations |
| **Cost per call** | Increases after many messages | Stays constant |

---

## Key Takeaways

### ✅ What We Learned:
1. **Summary memory** automatically compresses conversations
2. **Trigger-based summarization** happens when messages exceed a threshold
3. **Bounded memory** prevents costs from spiraling
4. **Important context** is preserved despite compression

### 🔧 Implementation Details:
- **Threshold**: 6 messages (configurable)
- **Summarization trigger**: Automatic when limit is reached
- **Process**: Generate summary, clear old messages, keep summary only

### ⚖️ Trade-offs:
- ✅ **Pros**: Lower costs, scalable for long conversations, bounded memory
- ❌ **Cons**: Possible information loss, slightly more complex logic

### 🚀 Advanced Variations:
- **ConversationWindowMemory**: Keep only last N messages (no summarization)
- **ConversationKGMemory**: Extract knowledge graph from conversations
- **Entity Memory**: Track specific entities across conversations
- **Hybrid Approach**: Summary + recent messages = best of both worlds

### 💡 Speaker Notes for Recap:
**Time: 3-4 minutes**

**Points to emphasize:**
1. This is the sweet spot for production systems - memory is bounded but meaningful
2. Show how the summarization prompt can be customized for different domains
3. Mention that the threshold (6 messages) is arbitrary and should be tuned for your use case
4. Contrast with buffer memory: "Buffer is great for understanding, Summary is great for scaling"
5. Real-world analogy: "Like reading a book - you remember the plot summary, not every word"

**Transition suggestion:**
"Now that we understand memory management, let's move to the next level: Tools and Agents. These let your AI do things, not just talk about them."

# ConversationWindowMemory with LangChain

## Learning Objectives
By the end of this section, you will understand:
- ✅ How ConversationWindowMemory differs from Buffer and Summary Memory
- ✅ Why a sliding window approach is useful for recent context
- ✅ How to keep only the last N messages efficiently
- ✅ Trade-offs between simplicity and context preservation

## What We'll Build
A chatbot that keeps only the most recent N messages, automatically dropping older messages to maintain a bounded memory footprint.

---

## Speaker Notes
**Key Talking Points:**
1. **The Middle Ground (2 min)**: WindowMemory sits between Buffer (keeps all) and Summary (compresses all)
2. **How It Works (3 min)**: Simple FIFO approach - just keep the last N messages
3. **Why Use It (2 min)**: Faster than summarization, but keeps more context than just a summary
4. **Trade-offs (2 min)**: Loses early context but maintains recent conversational flow
5. **Hands-on Demo (5 min)**: Show how old messages are automatically dropped

---

## What Window Memory Means (The Core Concept)

**Window memory keeps only the last N messages.**

It's like a sliding window that only shows the most recent part of the conversation.

**Key Benefits:**
- ⏱️ Very fast - no summarization needed
- 🎯 Keeps recent context intact
- 📊 Bounded memory size
- 💾 Simpler than summarization

**Trade-off:**
> You lose context from earlier in the conversation, but you maintain a clean, recent conversational flow.

In [ ]:
# Import the same components as before
from langchain_openai import ChatOpenAI  # LLM for conversation
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.runnables.history import RunnableWithMessageHistory  # Memory management
from langchain_community.chat_message_histories import ChatMessageHistory  # History storage

## Step 1: Import Required Libraries (Window Memory Version)

Same imports - the difference is in how we manage the message history:


In [35]:
# Initialize the language model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


## Step 2: Initialize the Language Model

Same as before:


In [36]:
# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Create the chain
chain = prompt | llm


## Step 3: Create Prompt and Chain

Standard prompt and chain setup:


In [38]:
# Create an in-memory store for conversation histories
store = {}

def get_session_history(session_id: str):
    """
    Retrieve or create a ChatMessageHistory for a session.
    This version implements a sliding window - keeps only the last N messages.
    
    Args:
        session_id (str): Unique identifier for the user/conversation
        
    Returns:
        ChatMessageHistory: The message history (with sliding window applied)
    """
    # Create a new history if this is the first message from this session
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    history = store[session_id]
    
    # KEY LOGIC: Implement sliding window - keep only last N messages
    # Window size: 4 messages (2 exchanges = user + AI response)
    WINDOW_SIZE = 4
    
    if len(history.messages) > WINDOW_SIZE:
        # Keep only the most recent WINDOW_SIZE messages
        # This automatically drops the oldest messages (FIFO)
        messages_to_keep = history.messages[-WINDOW_SIZE:]
        
        # Clear history and add back only the recent messages
        history.clear()
        for message in messages_to_keep:
            if message.__class__.__name__.startswith('Human'):
                history.add_user_message(message.content)
            else:
                history.add_ai_message(message.content)

    return history

# Create the chatbot with window memory
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)


## Step 4: Implement Window-Based Memory Storage (The Key Difference!)

**This is where ConversationWindowMemory differs:**

**How it works:**
1. Store all messages normally in a buffer
2. When messages exceed the window size (4 messages), trim from the beginning
3. Keep only the MOST RECENT N messages (sliding window)
4. Automatically drop the oldest messages

**Key Difference from Summarization:**
- ✅ **Faster**: No LLM call needed, just array slicing
- ✅ **Simpler**: Just keep last N, no complex logic
- ❌ **Context Loss**: Old messages are dropped, not summarized

**Window Size in This Example:**
- `WINDOW_SIZE = 4` means keep last 4 messages (2 exchanges)
- This is configurable for your use case


In [39]:
# Have 6 conversations to demonstrate the sliding window
# With WINDOW_SIZE=4, messages 1-2 will be dropped after we have 6 messages

# Message 1
response1 = chatbot.invoke(
    {"input": "My name is Alice"},
    config={"configurable": {"session_id": "user_window"}}
)
print("Message 1 - User: My name is Alice")
print(f"Assistant: {response1.content}\n")

# Message 2
response2 = chatbot.invoke(
    {"input": "I work in Berlin"},
    config={"configurable": {"session_id": "user_window"}}
)
print("Message 2 - User: I work in Berlin")
print(f"Assistant: {response2.content}\n")

# Message 3
response3 = chatbot.invoke(
    {"input": "I like Python programming"},
    config={"configurable": {"session_id": "user_window"}}
)
print("Message 3 - User: I like Python programming")
print(f"Assistant: {response3.content}\n")

# Message 4
response4 = chatbot.invoke(
    {"input": "I'm learning LangChain"},
    config={"configurable": {"session_id": "user_window"}}
)
print("Message 4 - User: I'm learning LangChain")
print(f"Assistant: {response4.content}\n")

print("="*70)
print("⚠️ Messages so far: 8 total (4 messages × 2 for user+AI)")
print("Window size is 4, so the oldest messages will be dropped next")
print("="*70 + "\n")

# Message 5 - This will trigger window trimming
response5 = chatbot.invoke(
    {"input": "My favorite framework is FastAPI"},
    config={"configurable": {"session_id": "user_window"}}
)
print("Message 5 - User: My favorite framework is FastAPI")
print(f"Assistant: {response5.content}\n")

print("="*70)
print("⚠️ Messages 1-2 were just dropped! Window is now Messages 3-5")
print("="*70 + "\n")

# Message 6
response6 = chatbot.invoke(
    {"input": "I'm building AI applications"},
    config={"configurable": {"session_id": "user_window"}}
)
print("Message 6 - User: I'm building AI applications")
print(f"Assistant: {response6.content}\n")


Message 1 - User: My name is Alice
Assistant: Nice to meet you, Alice! How can I assist you today?

Message 2 - User: I work in Berlin
Assistant: That's great! Berlin is a vibrant city with a rich history and a diverse culture. What do you do for work there?

Message 3 - User: I like Python programming
Assistant: That's awesome! Python is a versatile and powerful programming language. Are you working on any specific projects or areas in Python that you enjoy?

Message 4 - User: I'm learning LangChain
Assistant: That's exciting! LangChain is a framework designed to help developers build applications with language models. It provides tools for managing prompts, chaining together different components, and integrating with various data sources.

Are you focusing on any particular aspect of LangChain, such as building chatbots, data analysis, or something else? If you have any questions or need resources, feel free to ask!

⚠️ Messages so far: 8 total (4 messages × 2 for user+AI)
Window siz

## Step 5: Test - Demonstrate Sliding Window in Action

**What happens here:**
1. **Messages 1-4**: Buffer stores all messages (8 total)
2. **Message 5**: Window size exceeded → Messages 1-2 are dropped (only 4 most recent kept)
3. **Message 6**: Continues with only Messages 3-6 in memory

**The key difference:** No summarization! Just pure sliding window FIFO (First-In-First-Out)


In [40]:
# Test the window memory with a question
response = chatbot.invoke(
    {"input": "What do you know about me?"},
    config={"configurable": {"session_id": "user_window"}}
)

print("User: What do you know about me?")
print(f"\nAssistant:\n{response.content}")
print("\n" + "="*70)
print("✅ SUCCESS! Window Memory in Action!")
print("="*70)
print("\nKey Observation:")
print("- The AI remembers RECENT messages (Messages 3-6)")
print("- The AI FORGOT early messages (Messages 1-2 about name and location)")
print("- This demonstrates the sliding window effect!")
print("- Notice: No summarization, just a clean window of recent context")


User: What do you know about me?

Assistant:
I don't have any personal information about you unless you've shared something specific in this conversation. I only know that you enjoy using FastAPI and are building AI applications. If you have any specific questions or topics you'd like to discuss, feel free to share!

✅ SUCCESS! Window Memory in Action!

Key Observation:
- The AI remembers RECENT messages (Messages 3-6)
- The AI FORGOT early messages (Messages 1-2 about name and location)
- This demonstrates the sliding window effect!
- Notice: No summarization, just a clean window of recent context


## Step 6: Inspect the Window Store

See exactly what messages remain in the window:


In [ ]:
# Inspect the window store
print("="*70)
print("INSPECTING THE WINDOW MEMORY STORE")
print("="*70)
print(f"\nNumber of sessions in store: {len(store)}")
print(f"Session IDs: {list(store.keys())}")

# Get the history for user_window
user_window_history = store.get("user_window")
if user_window_history:
    print(f"\nNumber of messages in 'user_window' history: {len(user_window_history.messages)}")
    print("(Remember: WINDOW_SIZE = 4, so only 4 most recent messages kept)")
    print("\n" + "-"*70)
    print("MESSAGE CONTENT (In Window):")
    print("-"*70)
    
    for i, message in enumerate(user_window_history.messages, 1):
        msg_type = "User" if message.__class__.__name__.startswith('Human') else "AI"
        print(f"\nMessage {i} ({msg_type}):")
        print(f"{message.content}")
        print("-"*70)
    
    print("\n💡 Key Insights:")
    print("- Only 5 messages remain in the window (most recent ones)")
    print("- Compare to 12 total messages we would have with Buffer Memory")
    print("- Messages about 'Alice' and 'Berlin' (early messages) are GONE")
    print("- This is much FASTER than summarization - no LLM call needed!")
    print("- Trade-off: We lost early context, but kept recent conversational flow")
else:
    print("\nNo history found for user_window")


INSPECTING THE WINDOW MEMORY STORE

Number of sessions in store: 1
Session IDs: ['user_window']

Number of messages in 'user_window' history: 6
(Remember: WINDOW_SIZE = 4, so only 4 most recent messages kept)

----------------------------------------------------------------------
MESSAGE CONTENT (In Window):
----------------------------------------------------------------------

Message 1 (User):
My favorite framework is FastAPI
----------------------------------------------------------------------

Message 2 (AI):
FastAPI is a fantastic choice! It's a modern, fast (high-performance) web framework for building APIs with Python 3.6+ based on standard Python type hints. Some of its key features include:

- **Fast**: Very high performance, on par with Node.js and Go (thanks to Starlette and Pydantic).
- **Easy**: Designed to be easy to use and learn, with automatic interactive API documentation (Swagger UI and ReDoc).
- **Type Safety**: Utilizes Python type hints for data validation and s

## Comparison: Buffer Memory vs Summary Memory vs Window Memory

### Table Comparison

| Aspect | Buffer Memory | Summary Memory | Window Memory |
|--------|---------------|---|---|
| **Mechanism** | Keeps ALL messages | Summarizes old messages when threshold reached | Keeps only last N messages |
| **Memory Growth** | Linear (unbounded) | Bounded (summarization limit) | Bounded (fixed window size) |
| **Processing Overhead** | ✅ None - just storage | ⚠️ High - LLM call for summarization | ✅ Minimal - just array slicing |
| **Implementation Complexity** | ✅ Simple (no logic) | ⚠️ Complex (includes LLM prompt) | ✅ Simple (just keep last N) |
| **Early Context Quality** | ✅✅ Perfect - nothing lost | ⚠️ Good - summarized into brief | ❌ Lost - discarded after window |
| **Recent Context Quality** | ✅✅ Perfect - all details | ✅✅ Perfect - all details | ✅✅ Perfect - all details |
| **Token Usage Pattern** | Keeps growing → ❌ Expensive for long conversations | Stays bounded → ✅ Controlled | Stays bounded → ✅ Controlled |
| **Speed** | ✅ Fast (no computation) | ⚠️ Slower (LLM inference) | ✅ Fastest (simple slicing) |
| **When to Use** | Short conversations, where context matters | Long conversations, need detailed history | Real-time chat, context window limits |
| **Example Use Case** | Customer support (1-2 exchanges) | Deep research discussions | Multi-turn chatbots, mobile apps |
| **Max Tokens in Context** | No limit (grows with conversation) | ~500-2000 (configurable threshold) | Fixed (e.g., 4 messages = ~1000 tokens) |

### Trade-off Summary

**Choose Buffer Memory if:**
- Conversation is short (< 10 exchanges)
- Every detail matters (legal, medical, research)
- Cost is not a primary concern

**Choose Summary Memory if:**
- Need to keep detailed history for accuracy
- Conversation might be very long
- Have budget for occasional LLM summarization calls
- Balance needed between history and cost

**Choose Window Memory if:**
- Speed is critical (real-time requirements)
- Only recent context matters
- Want to avoid LLM computation overhead
- Client-side memory is limited (mobile apps)
- Don't care about early conversation history

### Example Scenarios

**Scenario 1: Customer Support Chatbot**
```
Conversation length: Typically 3-5 exchanges
Best choice: Buffer Memory ✅
Why: Short conversations, all context needed to resolve issue
```

**Scenario 2: Research Assistant**
```
Conversation length: 20-50+ exchanges
Best choice: Summary Memory ✅
Why: Long conversation, but user benefits from summarized research history
```

**Scenario 3: Mobile Chat Application**
```
Conversation length: 100+ exchanges, memory limited
Best choice: Window Memory ✅
Why: Recent messages sufficient, fast execution, low memory footprint
```

---



## Summary: Memory Management in LangChain

### What We've Learned

We've explored **three fundamental memory management strategies** in LangChain:

1. **ConversationBufferMemory** ✅ COMPLETED
   - Simplest approach: keep everything
   - Perfect context but unbounded growth
   - Best for short, context-sensitive conversations

2. **ConversationSummaryMemory** ✅ COMPLETED
   - Smart compression using LLM summarization
   - Bounded growth with good context preservation
   - Best for long-running, detail-critical conversations

3. **ConversationWindowMemory** ✅ COMPLETED
   - Fast sliding window approach
   - Fixed memory footprint, no LLM overhead
   - Best for real-time, recent-context-focused apps

### Key Takeaways

| What | Key Learning |
|------|--------------|
| **Memory Is Not Magic** | You MUST explicitly manage conversation history |
| **Trade-offs Everywhere** | Quality vs Speed vs Cost - pick your priorities |
| **Context Window Matters** | LLMs have token limits (4k-128k) - memory management is essential |
| **Implementation Pattern** | All three use same chain structure; only `get_session_history()` differs |
| **Testing Is Critical** | Always test with multiple messages to see threshold behavior |

### Advanced Variations You Can Explore

Once comfortable with these three types, consider:

- **Hybrid Memory**: Use Summary for older messages + Window for recent messages
- **Persistent Storage**: Replace in-memory dict with Redis, PostgreSQL, or MongoDB
- **Smart Summarization**: Summarize by topic instead of just compression
- **Filtered Memory**: Include/exclude certain message types (e.g., system messages)
- **Weighted Retrieval**: RAG-based memory that retrieves relevant past messages vs all messages

---

### 🎯 Speaker Notes (4-5 minutes)

**[Intro - 0:00-0:30]**
"You've just learned three ways to handle conversation memory in LangChain. Memory management is CRITICAL because LLMs have token limits, and conversations grow quickly. You can't just keep everything - you need strategy."

**[Buffer Memory - 0:30-1:15]**
"Buffer memory is the simplest. You keep every message. It's like a tape recorder - perfect fidelity, but the tape keeps getting longer. Great for short conversations where nothing can be lost."

**[Summary Memory - 1:15-2:15]**
"Summary memory is smarter. After you hit a threshold - let's say 6 messages - the system calls an LLM to create a brief summary of the conversation. Then replaces all those messages with the summary plus new messages. It's like having an assistant who reads long email threads and gives you the cliff notes. Takes a tiny bit longer due to the LLM call, but keeps your conversation bounded."

**[Window Memory - 2:15-3:15]**
"Window memory is the fastest. It's like a sliding window - you only keep the last 4 or 5 messages, and older ones fall off. No LLM call, no computation. Just pure array slicing. You lose early context, but who cares if you only need the last exchange for customer support on mobile?"

**[Trade-offs - 3:15-4:00]**
"So which one should you use? Ask yourself three questions: One, how long is the conversation? Two, how much does context matter? Three, what's your budget and speed requirement? These three memory types cover most real-world scenarios. And the pattern you learned - switching only the `get_session_history()` function - that's how you'll build production systems."

**[Closer - 4:00-4:15]**
"In the next section, we're moving to Tools and Agents - teaching your LLM to take actions. Memory management becomes even MORE critical there because agents often take many steps to solve a problem. See you in the next video!"

---

### 🚀 What's Next?

In the next section, we'll explore:
- **Tools**: Giving your LLM the ability to call external functions
- **Agents**: Building systems where LLMs decide what tools to use and when
- **Agent Memory**: How memory management works with multi-step reasoning

The memory patterns you learned here are ESSENTIAL foundations. Agents especially need careful memory management because they generate many intermediate thoughts and actions.

---
